In [1]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — saves to files, no display needed
import numpy as np
import matplotlib.pyplot as plt
import RK4_2D
import os

os.makedirs('output', exist_ok=True)


In [2]:
Lx = 10.0     # length of the domain (x-axis)
Ly = 20.0     # width of the domain (y-axis)
tend = 10.0

nx = 26      # grid points in x
ny = 51      # grid points in y
CFL = 0.5


dx = Lx/(nx-1)    # spatial step in x  (vertex-centred: last node at x=0 fault)
dy = Ly/(ny-1)    # spatial step in y  (vertex-centred: last node at y=Ly)

isnap = 5     # snapshot frequency

nf = 5        # number of fields
cs = 3.464    # shear wave speed
cp = 6.0      # compresional wave speed
rho = 2.6702  # density

# source parameters
x0 = -15.0        # [km]
y0 = 7.5         # [km]
t0 = 0.0         # [s]
T =  0.1         # [s]
M0 = 00.0      # [MPa]

source_type = 'Gaussian' # 'Gaussian', 'Brune'

# RS: rate-and-state friction law
# SW: slip-weakening friction law
fric_law = 'SW'   #RS OR SW

# fracture mode
# Mode II crack: In-plane shear
# Mode III crack: out-of-plane (anti-plane) shear
mode = 'II'   #  'II' or 'III'

# Mode II crack: In-plane shear
if mode == 'II':
    nf = 5        # number of fields [vx, vy, sxx, syy, sxy] 

# Mode III crack: out-of-plane (anti-plane) shear    
if mode == 'III':
    nf = 3        # number of fields  [vz, sxz, sxy]  


M = [0, 0, 1., 1., 0]

source_parameter = [x0, y0, t0, T, M0, source_type, M]

# extract Lame parameters
mu = rho*cs**2
Lambda = rho*cp**2-2.0*mu

dt = CFL/np.sqrt(cp**2 + cs**2)*dx    # Time step
nt = int(np.round(tend/dt))                             # number of time steps

order = 6        # spatial order of accuracy

# Model type, available are "homogeneous", "fault_zone",
# "surface_low_velocity_zone", "random", "topography",
# "slab"
model_type = "homogeneous"


In [3]:
# Initialize velocity model
Mat_l = np.zeros((nx, ny, 3))

Mat_r = np.zeros((nx, ny, 3))

if model_type == "homogeneous":
    Mat_l[:,:,0] += rho
    Mat_l[:,:,1] += Lambda
    Mat_l[:,:,2] += mu
    
    Mat_r[:,:,0] += rho
    Mat_r[:,:,1] += Lambda
    Mat_r[:,:,2] += mu

elif model_type == "random":
    pert_l = 0.4
    r_rho_l = 2.0 * (np.random.rand(nx, ny) - 0.5) * pert_l
    r_mu_l = 2.0 * (np.random.rand(nx, ny) - 0.5) * pert_l
    r_lambda_l = 2.0 * (np.random.rand(nx, ny) - 0.5) * pert_l    
    Mat_l[:,:,0] += rho*(1.0 + r_rho_l)
    Mat_l[:,:,1] += Lambda*(1.0 + r_lambda_l)
    Mat_l[:,:,2] += mu*(1.0 + r_mu_l)
    
    pert_r = 0.4
    r_rho_r = 2.0 * (np.random.rand(nx, ny) - 0.5) * pert_r
    r_mu_r = 2.0 * (np.random.rand(nx, ny) - 0.5) * pert_r
    r_rambda_r = 2.0 * (np.random.rand(nx, ny) - 0.5) * pert_r
    Mat_r[:,:,0] += rho*(1.0 + r_rho_r)
    Mat_r[:,:,1] += Lambda*(1.0 + r_rambda_r)
    Mat_r[:,:,2] += mu*(1.0 + r_mu_r)
    
# Initialize pressure at different time steps and the second
# derivatives in each direction
F_l = np.zeros((nx, ny, nf))
Fnew_l = np.zeros((nx, ny, nf))
X_l = np.zeros((nx, ny))
Y_l = np.zeros((nx, ny))
p_l = np.zeros((nx, ny))

F_r = np.zeros((nx, ny, nf))
Fnew_r = np.zeros((nx, ny, nf))
X_r = np.zeros((nx, ny))
Y_r = np.zeros((nx, ny))
p_r = np.zeros((nx, ny))




for i in range(0, nx):
    for j in range(0, ny):
        X_l[i,j] = -Lx + i*dx
        Y_l[i,j] = j*dy
        
        X_r[i,j] = i*dx
        Y_r[i,j] = j*dy
        
        #F_l[i,j,1] = -10**-12 
        #F_r[i,j,1] =  10**-12
        

Y_fault =np.zeros((ny, 1)) 
Y0 = 10

for j in range(0, ny):
        #Y_fault[j, 0] = j*dy
        
        if np.abs(j*dy-Y0) <= 1.5:
            Y_fault[j, 0] = 1.0


In [4]:
# FRICTION PARAMETERS

# Calculations and plottings for on fault slip velocity and stress


slip = np.zeros((ny, 1))
psi = np.zeros((ny, 1))
FaultOutput = np.zeros((ny, nt, 6)) # c
FaultOutput0 = np.zeros((ny, 6))
DomainOutput_l = np.zeros((nx, ny, nt, nf)) #
DomainOutput_r = np.zeros((nx, ny, nt, nf))

if fric_law not in ('SW', 'RS'):
    # Choose friction law: fric_law
    # We use linear (LN: T = alpha*v, alpha >=0)
    # Slip-weakening (SW)
    # Rate-and-state friction law (RS)
    
     print('friction law not implemented. choose fric_law = SW or fric_law = RS')
     exit(-1)
    
if fric_law  in ('SW'):
     
     slip = np.zeros((ny, 1))                        # initial slip (in m)
     slip_new = np.zeros((ny, 1))
     Tau_0 = np.ones((ny, 1))*(70+11.6*Y_fault)   
     alp_s = np.ones((ny, 1))*0.677                          # stastic friction
     alp_d = np.ones((ny, 1))*0.525                          # dynamic friction
     D_c = np.ones((ny, 1))*0.4                              # critical slip
     sigma_n = -np.ones((ny, 1))*120.0                        # normal stress 
     
        
     # These are not needed for the slip weakening case   
     psi = np.ones((ny, 1))*0.0                         # initial condition for the state variable in friction law
     psi_new = np.ones((ny, 1))*0.0
     L0 = np.ones((ny, 1))*1.0                               # state evolution distance
     f0 = np.ones((ny, 1))*1.0                               # referance friction coeff
     a = np.ones((ny, 1))*1.0                                # direct effect 
     b = np.ones((ny, 1))*1.0                                # evolution parameter 
     V0 = np.ones((ny, 1))*1.0                               # reference slip rate
     alpha = np.ones((ny, 1))*1e1000000                      # initial friction coefficient
        
     FaultOutput[:, 0, 2] = sigma_n[:,0]   
     FaultOutput[:, 0, 3] = Tau_0[:,0] 
     FaultOutput[:, 0, 4] = slip[:,0] 
     FaultOutput[:, 0, 5] = psi[:,0]   
    
if fric_law  in ('RS'):
     alpha = np.ones((ny, 1))*1e1000000                      # initial friction coefficient                                                                                   
     slip = np.ones((ny, 1))*0.0                         # initial slip (in m) 
     slip_new = np.zeros((ny, 1))
     #Tau_0 = np.ones((ny, 1))*81.24+0.1*0.36                 # initial load (81.24 in MPa), slight increase will unlock the fault   
     psi = np.ones((ny, 1))*0.4367                      # initial condition for the state variable in friction law
     psi_new = np.ones((ny, 1))*0.0
     L0 = np.ones((ny, 1))*0.02                              # state evolution distance
     f0 = np.ones((ny, 1))*0.6                               # referance friction coeff
     a = np.ones((ny, 1))*0.008                              # direct effect 
     b = np.ones((ny, 1))*0.012                              # evolution parameter 
     V0 = np.ones((ny, 1))*1.0e-6                            # reference slip rate
     sigma_n = -np.ones((ny, 1))*120.0                        # background normal stress 
     Tau_0 = np.ones((ny, 1))*75 #-2*0.2429*sigma_n*Y_fault
     Vin = np.ones((ny, 1))*2.0e-12 
     theta = L0/V0*np.exp(((a*np.log(2.0*np.sinh(75/(a*120)))-f0-a*np.log(Vin/V0))/b))
     psi[:,0] = f0[:,0] + b[:,0]*np.log(V0[:,0]/L0[:,0]*theta[:,0])
                          
    
     # These are not needed for the rate and state case   
     alp_s = np.ones((ny, 1))*1.0                             # stastic friction
     alp_d = np.ones((ny, 1))*1.0                             # dynamic friction
     D_c = np.ones((ny, 1))*1.0                               # critical slip
        
     FaultOutput[:, 0, 2] = sigma_n[:,0]  
     FaultOutput[:, 0, 3] = Tau_0[:,0]
     FaultOutput[:, 0, 4] = slip[:,0]
     FaultOutput[:, 0, 5] = psi[:,0] 
        
friction_parameters = np.zeros((12, ny))    
#friction_parameters = [alpha, alpha, Tau_0, L0, f0, a, b, V0, sigma_n, alp_s, alp_d, D_c]    
#                        0         1      2     3   4  5  6   7    8       9      10  11

for j in range(0, ny):
        friction_parameters[0, j] = alpha[j, 0]
        friction_parameters[1, j] = alpha[j, 0]
        friction_parameters[2, j] = Tau_0[j, 0]
        friction_parameters[3, j] = L0[j, 0]
        friction_parameters[4, j] = f0[j, 0]
        friction_parameters[5, j] = a[j, 0]
        
        
        friction_parameters[6, j] = b[j, 0]
        friction_parameters[7, j] = V0[j, 0]
        friction_parameters[8, j] = sigma_n[j, 0]
        friction_parameters[9, j] = alp_s[j, 0]
        friction_parameters[10, j] = alp_d[j, 0]
        friction_parameters[11, j] = D_c[j, 0]
        
        #if np.abs(j*dy-Y0) <= 1.5:
        #    Y_fault[j, 0] = 1.0
            
############ END FRICTION PARAMETERS


In [5]:
# Receiver locations left  (multi-station array)
rx_l = np.array([-9, -6, -3, 0, 0])
ry_l = np.array([0,   0,  0, 0, 10])

irx_l = np.zeros(len(rx_l), dtype=int)
iry_l = np.zeros(len(ry_l), dtype=int)

for i in range(len(rx_l)):
    irx_l[i] = int(np.ceil(rx_l[i]/dx)) + (nx - 1)
    iry_l[i] = int(np.ceil(ry_l[i]/dy))

seisvx_l = np.zeros((len(irx_l), nt))
seisvy_l = np.zeros((len(irx_l), nt))

# Receiver locations right  (multi-station array)
rx_r = np.array([9, 6, 3, 0, 0])
ry_r = np.array([0, 0, 0, 0, 10])

irx_r = np.zeros(len(rx_r), dtype=int)
iry_r = np.zeros(len(ry_r), dtype=int)

for i in range(len(rx_r)):
    irx_r[i] = int(np.ceil(rx_r[i]/dx))
    iry_r[i] = int(np.ceil(ry_r[i]/dy))

seisvx_r = np.zeros((len(irx_r), nt))
seisvy_r = np.zeros((len(irx_r), nt))



# Boundary reflection coefficients: 0<= r[j] <= 1
r_l = np.array([0.,0.,1.,0.])
r_r = np.array([0.,0.,1.,0.])

# required for seismograms
ir_l = np.arange(len(irx_l))
ir_r = np.arange(len(irx_r))


print(irx_l)
print(iry_l)

#exit()


[ 3 10 18 25 25]
[ 0  0  0  0 25]


In [6]:
#################################################

# Time-stepping 
for it in range(nt):
    
    t = it*dt
    #4th order Runge-Kutta 
    RK4_2D.elastic_RK4_2D(Fnew_l, F_l, Mat_l, X_l, Y_l, t, nf, nx, ny, dx, dy, dt, order, r_l, source_parameter, Fnew_r, F_r, Mat_r, X_r, Y_r, r_r, friction_parameters, slip,  psi, slip_new, psi_new, fric_law, FaultOutput0, Y0)

    #(DF_l, F_l, Mat_l, X_l, Y_l, t, nf, nx, ny, dx, dy, dt, order, r_l, source_parameter, DF_r, F_r, Mat_r, X_r, Y_r, r_r,friction_parameters, slip, psi, DF_slip, DF_psi, fric_law, FaultOutput0, Y0):
    # update fields
    F_l = Fnew_l
    F_r = Fnew_r
    slip = slip_new 
    psi = psi_new
    
    FaultOutput[:, it, :] = FaultOutput0
    DomainOutput_l[:, :, it, :] = F_l
    DomainOutput_r[:, :, it, :] = F_r

    
    #extract particle velocity for visualization
    #p_l = np.sqrt(F_l[:,:,0]**2 + F_l[:,:,1]**2)
    #p_r = np.sqrt(F_r[:,:,0]**2 + F_r[:,:,1]**2)
    
    # update time
    t = it*dt
    
    if it % isnap == 0:
        
        print('time =', t, 'iteration count =', it)
    
    p_l = F_l[:,:,1]
    p_r = F_r[:,:,1]
    
    

    
    if mode == 'II':
        # Save seismograms
        seisvx_l[ir_l, it] = F_l[irx_l[ir_l], iry_l[ir_l], 0]
        seisvy_l[ir_l, it] = F_l[irx_l[ir_l], iry_l[ir_l], 1]
    
        seisvx_r[ir_r, it] = F_r[irx_r[ir_r], iry_r[ir_r], 0]
        seisvy_r[ir_r, it] = F_r[irx_r[ir_r], iry_r[ir_r], 1]
        

    if mode == 'III':
        # Save seismograms
        seisvx_l[ir_l, it] = F_l[irx_l[ir_l], iry_l[ir_l], 0]
        seisvx_r[ir_r, it] = F_r[irx_r[ir_r], iry_r[ir_r], 0]
        
    


time = 0.0 iteration count = 0
time = 0.14433862578454332 iteration count = 5
time = 0.28867725156908663 iteration count = 10
time = 0.43301587735362995 iteration count = 15
time = 0.5773545031381733 iteration count = 20
time = 0.7216931289227166 iteration count = 25
time = 0.8660317547072599 iteration count = 30
time = 1.0103703804918032 iteration count = 35
time = 1.1547090062763465 iteration count = 40
time = 1.29904763206089 iteration count = 45
time = 1.4433862578454333 iteration count = 50
time = 1.5877248836299764 iteration count = 55
time = 1.7320635094145198 iteration count = 60
time = 1.8764021351990632 iteration count = 65
time = 2.0207407609836063 iteration count = 70
time = 2.16507938676815 iteration count = 75
time = 2.309418012552693 iteration count = 80
time = 2.4537566383372362 iteration count = 85
time = 2.59809526412178 iteration count = 90
time = 2.742433889906323 iteration count = 95
time = 2.8867725156908666 iteration count = 100
time = 3.0311111414754097 iteratio

In [9]:
from matplotlib.animation import FuncAnimation, PillowWriter

# ── Wavefield animation ────────────────────────────────────────────────────
fig_anim, ax_anim = plt.subplots(figsize=(8, 5))

p_l0 = DomainOutput_l[:, :, 0, 1] if mode == 'II' else DomainOutput_l[:, :, 0, 0]
p_r0 = DomainOutput_r[:, :, 0, 1] if mode == 'II' else DomainOutput_r[:, :, 0, 0]
p_b0 = np.squeeze(np.append([p_l0.transpose()], [p_r0.transpose()], axis=2))

v = 2.0
im = ax_anim.imshow(p_b0, aspect='auto', extent=[-Lx, Lx, Ly, 0],
                    cmap='seismic', vmin=-v, vmax=+v, interpolation='none')
for x, y in zip(rx_l, ry_l):
    ax_anim.text(x, y, '+', ha='center', va='center', fontsize=8)
for x, y in zip(rx_r, ry_r):
    ax_anim.text(x, y, '+', ha='center', va='center', fontsize=8)
ax_anim.text(x0, y0, 'o', ha='center', va='center', fontsize=8)
fig_anim.colorbar(im, ax=ax_anim)
ax_anim.set_xlabel('x [km]')
ax_anim.set_ylabel('y [km]')
title = ax_anim.set_title('time: 0.00 s')

anim_frames = range(0, nt, max(1, nt // 100))   # ~100 frames max

def _update_wave(it):
    if mode == 'II':
        p_l = DomainOutput_l[:, :, it, 1]
        p_r = DomainOutput_r[:, :, it, 1]
    else:
        p_l = DomainOutput_l[:, :, it, 0]
        p_r = DomainOutput_r[:, :, it, 0]
    p_b = np.squeeze(np.append([p_l.transpose()], [p_r.transpose()], axis=2))
    im.set_data(p_b)
    title.set_text(f'time: {it * dt:.2f} s')
    return [im, title]

ani = FuncAnimation(fig_anim, _update_wave, frames=anim_frames, blit=True)
ani.save('output/rupture_2d_SW_wavefield.gif', writer=PillowWriter(fps=15))
print("Saved: output/rupture_2d_SW_wavefield.gif")
plt.close(fig_anim)

# ── Seismograms ────────────────────────────────────────────────────────────
fig4 = plt.figure(figsize=(10, 5))
time = np.arange(nt) * dt

if mode == 'II':
    ay1 = fig4.add_subplot(2, 2, 1)
    ymax = max(np.abs(seisvx_l).max(), 1e-10)
    for i in range(len(seisvx_l)):
        ay1.plot(time, seisvx_l[i, :] + ymax * i)
    ay1.set_ylabel('vx (m/s)')
    ay1.set_xlim([0, tend])
    ay1.set_title('Left domain — vx')

    ay2 = fig4.add_subplot(2, 2, 2)
    ymax = max(np.abs(seisvy_l).max(), 1e-10)
    for i in range(len(seisvy_l)):
        ay2.plot(time, seisvy_l[i, :] + ymax * i)
    ay2.set_ylabel('vy (m/s)')
    ay2.set_xlim([0, tend])
    ay2.set_title('Left domain — vy')

    ay3 = fig4.add_subplot(2, 2, 3)
    ymax = max(np.abs(seisvx_r).max(), 1e-10)
    for i in range(len(seisvx_r)):
        ay3.plot(time, seisvx_r[i, :] + ymax * i)
    ay3.set_xlabel('Time (s)')
    ay3.set_ylabel('vx (m/s)')
    ay3.set_xlim([0, tend])
    ay3.set_title('Right domain — vx')

    ay4 = fig4.add_subplot(2, 2, 4)
    ymax = max(np.abs(seisvy_r).max(), 1e-10)
    for i in range(len(seisvy_r)):
        ay4.plot(time, seisvy_r[i, :] + ymax * i)
    ay4.set_xlabel('Time (s)')
    ay4.set_ylabel('vy (m/s)')
    ay4.set_xlim([0, tend])
    ay4.set_title('Right domain — vy')

else:  # mode III
    ax1 = fig4.add_subplot(2, 1, 1)
    ymax = max(np.abs(seisvx_l).max(), 1e-10)
    for i in range(len(seisvx_l)):
        ax1.plot(time, seisvx_l[i, :] + ymax * i)
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('vz left (m/s)')

    ax2 = fig4.add_subplot(2, 1, 2)
    ymax = max(np.abs(seisvx_r).max(), 1e-10)
    for i in range(len(seisvx_r)):
        ax2.plot(time, seisvx_r[i, :] + ymax * i)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('vz right (m/s)')

fig4.tight_layout()
fig4.savefig('output/rupture_2d_SW_seismograms.png', dpi=150)
print("Saved: output/rupture_2d_SW_seismograms.png")
plt.close(fig4)


Saved: output/rupture_2d_SW_wavefield.gif
Saved: output/rupture_2d_SW_seismograms.png


In [10]:
from matplotlib.animation import FuncAnimation, PillowWriter

y_fault = Y_l[-1, :]

fig1, (ax3, ax4, ax5) = plt.subplots(3, 1, figsize=(10, 10))
ax3.set_ylabel('Slip [m]');      ax3.set_ylim([0, 20]);   ax3.set_xlim([0, Ly])
ax4.set_ylabel('Slip rate [m/s]'); ax4.set_ylim([0, 7]);  ax4.set_xlim([0, Ly])
ax5.set_ylabel('stress [MPa]'); ax5.set_xlabel('fault [km]')
ax5.set_ylim([50, 90]);          ax5.set_xlim([0, Ly])

line3, = ax3.plot([], [], 'g', lw=2, label='slip')
line4, = ax4.plot([], [], 'r', lw=2, label='slip rate')
line5, = ax5.plot([], [], 'b', lw=2, label='traction')
ax3.legend(); ax4.legend(); ax5.legend()
fig1.tight_layout()
sup = fig1.suptitle('t = 0.00 s', y=1.01)

fault_frames = range(0, nt, max(1, nt // 100))   # ~100 frames

def _update_fault(it):
    slip_      = FaultOutput[:, it, 4]
    sliprate_  = np.sqrt(FaultOutput[:, it, 0]**2 + FaultOutput[:, it, 1]**2)
    traction_  = FaultOutput[:, it, 3]
    line3.set_data(y_fault, slip_)
    line4.set_data(y_fault, sliprate_)
    line5.set_data(y_fault, traction_)
    sup.set_text(f't = {it * dt:.2f} s')
    return line3, line4, line5

ani_fault = FuncAnimation(fig1, _update_fault, frames=fault_frames, blit=False)
ani_fault.save('output/rupture_2d_SW_fault_output.gif', writer=PillowWriter(fps=10))
print("Saved: output/rupture_2d_SW_fault_output.gif")
plt.close(fig1)


Saved: output/rupture_2d_SW_fault_output.gif


In [11]:
VT = np.sqrt(FaultOutput[:, :, 0]**2 + FaultOutput[:, :, 1]**2).T   # shape (nt, ny)

v = 2.5
fig_vt, ax_vt = plt.subplots(figsize=(8, 5))
im = ax_vt.imshow(VT, aspect='auto', extent=[0, Ly, nt * dt, 0],
                  cmap='viridis', vmin=0, vmax=v, interpolation='none')
fig_vt.colorbar(im, ax=ax_vt, label='slip rate [m/s]')
ax_vt.set_xlabel('fault [km]')
ax_vt.set_ylabel('t [s]')
ax_vt.set_title('On-fault slip rate  —  SW')
fig_vt.tight_layout()
fig_vt.savefig('output/rupture_2d_SW_sliprate_spacetime.png', dpi=150)
print("Saved: output/rupture_2d_SW_sliprate_spacetime.png")
plt.close(fig_vt)


Saved: output/rupture_2d_SW_sliprate_spacetime.png
